# V6.5-12 — YOLO11n@640 Roboflow AdamW + Cosine Ablation

Run này giữ nguyên dataset, model khởi tạo và seed của V6.5-11; chỉ thay optimizer/scheduler để so sánh sạch với MuSGD. Train trực tiếp trên GPU WSL, không ghi đè run cũ.


In [ ]:
# 0. CONFIG
from pathlib import Path
from datetime import datetime, timezone
import json

PROJECT = Path("/mnt/c/Users/intern/Documents/SEP490/train")
DATASET = Path.home() / "datasets/v6p5_roboflow_ablation"
DATA_YAML = DATASET / "data.yaml"
MODEL = PROJECT / "legacy/notebooks/v6p5/yolo11n.pt"
PREV_BEST = PROJECT / (
    "artifacts/v6p5_train_local/20261009T024019Z_roboflow/"
    "runs/yolo11n_v6p5_roboflow_ablation_640/weights/best.pt"
)

NAMES = ["insulator", "insulator_broken", "insulator_flashover",
         "bird_nest", "foreign_object", "broken_strand"]
IMGSZ = 640
EPOCHS = 75
BATCH = 8
DEVICE = 0
WORKERS = 0
SEED = 20261007
RUN_NAME = "yolo11n_v6p5_roboflow_adamw_cosine_640"

# Ghim RUN_ID này nếu cần resume sau khi restart kernel.
RUN_ID = None
RUN_ID = RUN_ID or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_adamw"
ROOT = PROJECT / "artifacts/v6p5_train_local" / RUN_ID
RUN_DIR = ROOT / "runs" / RUN_NAME
ROOT.mkdir(parents=True, exist_ok=True)

for required in [DATA_YAML, MODEL, PREV_BEST]:
    if not required.is_file():
        raise FileNotFoundError(required)
print("RUN_ID:", RUN_ID)
print("Dataset:", DATA_YAML)
print("Output:", RUN_DIR)


In [ ]:
# 1. GPU + DATASET GATE (NO TRAINING)
import yaml
import torch
import ultralytics

if not torch.cuda.is_available():
    raise RuntimeError("CUDA chưa khả dụng trong kernel WSL.")
if not 0 <= DEVICE < torch.cuda.device_count():
    raise ValueError(f"DEVICE={DEVICE} không hợp lệ.")
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(
        f"Cần ultralytics==8.4.172 để so sánh công bằng; hiện tại={ultralytics.__version__}"
    )
cfg = yaml.safe_load(DATA_YAML.read_text())
cfg_names = cfg.get("names", {})
dataset_names = ([cfg_names[k] for k in sorted(cfg_names, key=int)]
                 if isinstance(cfg_names, dict) else list(cfg_names))
if dataset_names != NAMES:
    raise ValueError(f"Taxonomy mismatch: {dataset_names}")
dataset_root = Path(cfg["path"])
for split in ["train", "val"]:
    image_dir = dataset_root / cfg[split]
    if not image_dir.is_dir():
        raise FileNotFoundError(image_dir)
    print(split, len([p for p in image_dir.iterdir() if p.is_file()]))
print("GPU:", torch.cuda.get_device_name(DEVICE))
print("PyTorch:", torch.__version__, "| Ultralytics:", ultralytics.__version__)


In [ ]:
# 2. TRAIN / RESUME — ADAMW + COSINE
from ultralytics import YOLO

last = RUN_DIR / "weights/last.pt"
if last.is_file() and last.stat().st_size > 0:
    print("RESUME:", last)
    model = YOLO(str(last))
    model.train(resume=True, device=DEVICE, workers=WORKERS)
else:
    print("NEW ADAMW RUN from:", MODEL)
    model = YOLO(str(MODEL))
    model.train(
        data=str(DATA_YAML),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        device=DEVICE,
        workers=WORKERS,
        optimizer="AdamW",
        lr0=0.001,
        lrf=0.01,
        weight_decay=0.01,
        cos_lr=True,
        warmup_epochs=3.0,
        patience=15,
        pretrained=True,
        seed=SEED,
        deterministic=True,
        cache=False,
        save=True,
        save_period=5,
        project=str(ROOT / "runs"),
        name=RUN_NAME,
        exist_ok=True,
        plots=False,
        verbose=True,
    )
best = RUN_DIR / "weights/best.pt"
if not best.is_file():
    raise FileNotFoundError(best)
print("Best:", best)
print("Last:", last)


In [ ]:
# 3. FAIR EVAL — ADAMW VS MUSGD ON THE SAME FROZEN VAL
import gc
import pandas as pd

try:
    model.cpu()
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

rows = []
for tag, weights in [("adamw_cosine", best), ("musgd_auto", PREV_BEST)]:
    metrics = YOLO(str(weights)).val(
        data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH,
        device=DEVICE, workers=WORKERS, plots=False, verbose=False,
        project=str(ROOT / "eval"), name=tag, exist_ok=True,
    ).box
    rows.append({"model": tag, "class": "ALL",
                 "AP50": float(metrics.map50), "AP50_95": float(metrics.map)})
    for cid, ap50, ap in zip(metrics.ap_class_index, metrics.ap50, metrics.ap):
        rows.append({"model": tag, "class": NAMES[int(cid)],
                     "AP50": float(ap50), "AP50_95": float(ap)})
result = pd.DataFrame(rows)
result.to_csv(ROOT / "adamw_vs_musgd_eval.csv", index=False)
display(result.pivot_table(index="class", columns="model",
                           values=["AP50", "AP50_95"]).round(4))


In [ ]:
# 4. DECISION GATE
wide = result.pivot(index="class", columns="model", values=["AP50", "AP50_95"])
def delta(metric, cls):
    return float(wide.loc[cls, (metric, "adamw_cosine")]
                 - wide.loc[cls, (metric, "musgd_auto")])

deltas = {
    "global_AP50": delta("AP50", "ALL"),
    "global_AP50_95": delta("AP50_95", "ALL"),
    "foreign_AP50_95": delta("AP50_95", "foreign_object"),
    "strand_AP50_95": delta("AP50_95", "broken_strand"),
}
gate = {
    "global_not_worse_0p5pt": deltas["global_AP50_95"] >= -0.005,
    "global_improves": deltas["global_AP50_95"] > 0.0,
    "foreign_not_worse": deltas["foreign_AP50_95"] >= 0.0,
    "strand_not_worse": deltas["strand_AP50_95"] >= 0.0,
}
decision = {"run_id": RUN_ID, "deltas_vs_musgd": deltas,
            "gate": gate, "pass": all(gate.values())}
(ROOT / "adamw_gate.json").write_text(json.dumps(decision, indent=2))
print(json.dumps(decision, indent=2))
print("PASS" if decision["pass"] else "FAIL")


## Cách chạy

Chạy từ cell 0 xuống. Nếu kernel bị ngắt, đặt `RUN_ID` đúng giá trị đã in rồi chạy lại cell 0, 1 và cell TRAIN; cell TRAIN sẽ resume `last.pt`. Không dùng checkpoint MuSGD để khởi tạo run AdamW.
